# Module 35 — Exercise 1: Circuit Breaker State Machine

In distributed systems, cascading failures occur when services endlessly wait on failing dependencies. A Circuit Breaker fails fast when a threshold of failures occurs, allowing the downstream service time to recover.

In this exercise, you will implement a 3-state Circuit Breaker (`CLOSED`, `OPEN`, `HALF_OPEN`).

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 35 README |



# Your turn


### Task 1: Circuit Breaker Implementation

Implement `CircuitBreaker`:
- State starts `CLOSED`. If consecutive failures >= `failure_threshold`, transition to `OPEN` and record `opened_at`.
- When in `OPEN`: if `now - opened_at > recovery_timeout`, transition to `HALF_OPEN`. Otherwise, raise `CircuitBreakerOpenError` immediately.
- When in `HALF_OPEN`: allow one trial call. If it succeeds, reset to `CLOSED`. If it fails, transition back to `OPEN`.


In [ ]:
# ANSWER 1
import time

class CircuitBreakerOpenError(Exception):
    pass

class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, recovery_timeout: float = 5.0):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.state = "CLOSED"
        self.failure_count = 0
        self.opened_at = 0.0

    def call(self, fn, *args, **kwargs):
        now = time.time()
        if self.state == "OPEN":
            if now - self.opened_at > self.recovery_timeout:
                self.state = "HALF_OPEN"
            else:
                raise CircuitBreakerOpenError("Circuit is OPEN: Failing fast")

        try:
            result = fn(*args, **kwargs)
            if self.state == "HALF_OPEN":
                self.state = "CLOSED"
                self.failure_count = 0
            return result
        except Exception:
            self.failure_count += 1
            if self.failure_count >= self.failure_threshold or self.state == "HALF_OPEN":
                self.state = "OPEN"
                self.opened_at = time.time()
            raise



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

cb = CircuitBreaker(failure_threshold=2, recovery_timeout=0.1)

def failing_service():
    raise RuntimeError("503 Service Unavailable")

def healthy_service():
    return "ok"

# Trigger 2 failures to trip circuit
try: cb.call(failing_service)
except RuntimeError: pass

try: cb.call(failing_service)
except RuntimeError: pass

state_tripped = cb.state

# Immediate next call should fail-fast with CircuitBreakerOpenError
tripped_fast = False
try:
    cb.call(healthy_service)
except CircuitBreakerOpenError:
    tripped_fast = True

time.sleep(0.15)  # Wait for recovery timeout

# Next call should trial in HALF_OPEN and recover to CLOSED
recovered = cb.call(healthy_service)
state_recovered = cb.state

results = [
    check(state_tripped == "OPEN", "Task 1: Circuit tripped to OPEN after 2 failures"),
    check(tripped_fast is True, "Task 1: Open circuit failed fast without executing service function"),
    check(recovered == "ok" and state_recovered == "CLOSED", "Task 1: Recovered cleanly to CLOSED after successful half-open trial"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

